# Wikipedia RAG with Chroma and OpenAI

This notebook shows a simple Retrieval-Augmented Generation flow:

1. load public Wikipedia articles
2. split them into chunks
3. create embeddings
4. upsert them into a local Chroma collection
5. retrieve top-k chunks with semantic search
6. send that context to OpenAI for answering

Run the notebook top to bottom the first time.

In [16]:
import ast
import os
import warnings
import zipfile
from pathlib import Path

warnings.filterwarnings("ignore")

import chromadb
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path("..").resolve()
load_dotenv(PROJECT_ROOT / ".env")

True

In [30]:
DATA_DIR = Path("../data")
ZIP_PATH = DATA_DIR / "lesson2-wiki.csv.zip"
CSV_PATH = DATA_DIR / "wiki.csv"
ROW_LIMIT = None
UPSERT_BATCH_SIZE = 5000
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "wikipedia_rag"
DB_PATH = Path("../chroma_db")
OPENAI_MODEL = "gpt-4o-mini"
RETRIEVAL_TOP_K = 2
MAX_CHUNK_CHARS = 500
MAX_OUTPUT_TOKENS = 250

In [31]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    !wget -q -O {ZIP_PATH} "https://www.dropbox.com/scl/fi/yxzmsrv2sgl249zcspeqb/lesson2-wiki.csv.zip?rlkey=paehnoxjl3s5x53d1bedt4pmc&dl=0"
else:
    print(f"Using existing {ZIP_PATH.name}")

print(f"Dataset archive: {ZIP_PATH.stat().st_size / 1e6:.1f} MB")

Using existing lesson2-wiki.csv.zip
Dataset archive: 142.5 MB


In [32]:
if not CSV_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_DIR)

raw_df = pd.read_csv(CSV_PATH)
if ROW_LIMIT:
    raw_df = raw_df.head(ROW_LIMIT)

metadata_df = raw_df["metadata"].apply(ast.literal_eval).apply(pd.Series)

chunks_df = pd.DataFrame(
    {
        "id": raw_df["id"].astype(str),
        "wiki_id": metadata_df["wiki-id"],
        "title": metadata_df["title"],
        "source": metadata_df["source"],
        "chunk_index": metadata_df["chunk"],
        "text": metadata_df.apply(
            lambda row: f"Title: {row['title']}\n\n{row['text']}",
            axis=1,
        ),
    }
)

article_count = chunks_df["wiki_id"].nunique()
print(f"Loaded {len(chunks_df):,} chunks from {article_count:,} Wikipedia articles")
chunks_df.head()

Loaded 10,000 chunks from 2,949 Wikipedia articles


,id,wiki_id,title,source,chunk_index,text
1,1-0,1,April,https://simple.wikipedia.org/wiki/April,0,Title: April\n\nApril is the fourth month of t...
2,1-1,1,April,https://simple.wikipedia.org/wiki/April,1,Title: April\n\nIn years immediately before co...
3,1-2,1,April,https://simple.wikipedia.org/wiki/April,2,Title: April\n\nApril 1 - April Fools' Day\n A...
4,1-3,1,April,https://simple.wikipedia.org/wiki/April,3,Title: April\n\nApril 15 - Father Damien Day (...
5,1-4,1,April,https://simple.wikipedia.org/wiki/April,4,Title: April\n\nApril 24 - Republic Day (the G...


In [33]:
def show_document_chunks(title=None, wiki_id=None):
    if title is None and wiki_id is None:
        raise ValueError("Pass either title or wiki_id")

    if wiki_id is not None:
        doc_chunks = chunks_df[chunks_df["wiki_id"] == wiki_id]
    else:
        doc_chunks = chunks_df[chunks_df["title"] == title]

    if doc_chunks.empty:
        raise ValueError(f"No document found for title={title!r}, wiki_id={wiki_id!r}")

    doc_chunks = doc_chunks.sort_values("chunk_index")
    header = doc_chunks.iloc[0]

    for _, row in doc_chunks.iterrows():
        print("=" * 80)
        print(f"Chunk {row['chunk_index']}  (id: {row['id']})")
        print("=" * 80)
        print(row["text"])
        print()

    return doc_chunks[["id", "title", "chunk_index"]]


# Example: all chunks for the "April" article
show_document_chunks(title="April")

Chunk 0  (id: 1-0)
Title: April

April is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.

April always begins on the same day of week as July, and additionally, January in leap years. April always ends on the same day of the week as December.

April's flowers are the Sweet Pea and Daisy. Its birthstone is the diamond. The meaning of the diamond is innocence.

The Month 

April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.

April begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other's last days are exactly 35 weeks (245 days) apart.

In common years, April starts on the same day of the week as October of the pr

,id,title,chunk_index
1,1-0,April,0
2,1-1,April,1
3,1-2,April,2
4,1-3,April,3
5,1-4,April,4
6,1-5,April,5
7,1-6,April,6
8,1-7,April,7
9,1-8,April,8
10,1-9,April,9


In [34]:
embedding_model = SentenceTransformer(MODEL_NAME)
embeddings = embedding_model.encode(
    chunks_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embeddings.shape

Batches: 100%|██████████| 157/157 [00:47<00:00,  3.28it/s]


(10000, 384)

In [35]:
client = chromadb.PersistentClient(path=str(DB_PATH))

try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

ids = chunks_df["id"].tolist()
documents = chunks_df["text"].tolist()
embedding_list = embeddings.tolist()
metadatas = [
    {"title": title, "chunk_index": int(chunk_index)}
    for title, chunk_index in zip(chunks_df["title"], chunks_df["chunk_index"])
]

for start in range(0, len(ids), UPSERT_BATCH_SIZE):
    end = start + UPSERT_BATCH_SIZE
    collection.upsert(
        ids=ids[start:end],
        documents=documents[start:end],
        embeddings=embedding_list[start:end],
        metadatas=metadatas[start:end],
    )
    print(f"Upserted {min(end, len(ids)):,} / {len(ids):,}")

collection.count()

Upserted 5,000 / 10,000
Upserted 10,000 / 10,000


10000

In [36]:
def retrieve_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
    )

    rows = []
    for doc_id, doc, meta, distance in zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        rows.append(
            {
                "id": doc_id,
                "title": meta["title"],
                "chunk_index": meta["chunk_index"],
                "distance": distance,
                "text": doc,
            }
        )

    return pd.DataFrame(rows)

In [37]:
retrieve_chunks("How do black holes form?", top_k=5)[["title", "chunk_index", "distance", "text"]]

,title,chunk_index,distance,text
0,A Brief History of Time,7,0.380892,Title: A Brief History of Time\n\nBlack Holes ...
1,Black hole,3,0.425110,"Title: Black hole\n\nAs of spring 2019, there ..."
2,Black hole,0,0.460657,Title: Black hole\n\nA black hole is a region ...
3,Black hole,5,0.514304,Title: Black hole\n\nSince we cannot see black...
4,Black hole,4,0.514849,Title: Black hole\n\nHuge central masses (106 ...


In [38]:
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        f"OPENAI_API_KEY not found. Create {PROJECT_ROOT / '.env'} with:\n"
        "OPENAI_API_KEY=sk-...\n"
        "Then restart the kernel and re-run the imports cell."
    )

openai_client = OpenAI()

In [39]:
def truncate_text(text, max_chars=MAX_CHUNK_CHARS):
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."


def answer_question(
    query: str,
    top_k: int = RETRIEVAL_TOP_K,
    model: str = OPENAI_MODEL,
    max_output_tokens: int = MAX_OUTPUT_TOKENS,
):
    retrieved = retrieve_chunks(query, top_k=top_k)

    context_blocks = []
    for idx, row in retrieved.iterrows():
        snippet = truncate_text(row["text"])
        context_blocks.append(
            f"[Source {idx + 1}] Title: {row['title']}\n"
            f"Chunk: {row['chunk_index']}\n"
            f"Content:\n{snippet}"
        )

    context = "\n\n".join(context_blocks)
    approx_input_chars = len(context) + len(query)

    prompt = f"""
You are a helpful assistant answering questions from retrieved Wikipedia context.

Rules:
- Use only the context below.
- If the context is insufficient, say: 'I do not know based on the retrieved context.'
- Keep the answer to 2-4 sentences.
- Mention source titles when useful.

Question:
{query}

Context:
{context}
""".strip()

    print(
        f"Retrieved {len(retrieved)} chunks | "
        f"~{approx_input_chars:,} input chars | "
        f"max {max_output_tokens} output tokens"
    )

    response = openai_client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=max_output_tokens,
    )

    return response.output_text, retrieved

In [40]:
answer, retrieved = answer_question("How do black holes form?")
print(answer)
retrieved[["title", "chunk_index", "distance"]]

Retrieved 2 chunks | ~1,132 input chars | max 250 output tokens


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}